## 1. Data Acquisition

**Source:** UCI Machine Learning Repository — "Online Retail" dataset, donated by Daqing Chen (School of Engineering, London South Bank University), 2015.

**Context:** Transactional records from a UK-based, non-store online retailer between 01/12/2010 and 09/12/2011. The company sells all-occasion gifts, and many customers are wholesalers rather than individual consumers.

**Relevance:** This dataset was chosen to explore customer segmentation and retention — a core marketing/CRM problem. Given my marketing and AI-automation background, I selected it to practice translating raw transactional data into insights that could inform real business decisions around customer retention, high-value customer identification, and targeted marketing strategy.

## 2. Data Structure

The raw file is a single flat table (one row per product line within an invoice), representing three underlying entities:
- **Customer** — CustomerID, Country
- **Invoice (Transaction)** — InvoiceNo, InvoiceDate, belongs to one Customer
- **Product** — StockCode, Description, UnitPrice

**Relationships:** One Customer → many Invoices; one Invoice → many product lines (each with its own Quantity); one Product can appear across many Invoices.

In [2]:
import pandas as pd
df = pd.read_excel('Online Retail.xlsx')
df.shape

(541909, 8)

This confirms the dataset contains 541,909 rows and 8 columns.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [6]:
df.isnull().sum()
df.duplicated().sum()

np.int64(5268)

In [8]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(5268)

In [10]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## 3. Initial Data Profiling

| Metric | Value |
|---|---|
| Total rows | 541,909 |
| Total columns | 8 |
| Missing Description | 1,454 rows |
| Missing CustomerID | 135,080 rows (24.9%) |
| Duplicate rows | 5,268 |
| Quantity range | -80,995 to 80,995 (avg 9.55) |
| UnitPrice range | -£11,062.06 to £38,970.00 (avg £4.61) |
| InvoiceDate range | 01 Dec 2010 – 09 Dec 2011 |

## 4. Data Quality Assessment

| Dimension | Observation |
|---|---|
| **Completeness** | 24.9% of rows are missing CustomerID — likely guest/non-account purchases. Limits customer-level analysis (segmentation, retention) to the ~75% of rows with an identified customer, but doesn't affect product- or transaction-level analysis. |
| **Accuracy** | UnitPrice contains a minimum of -£11,062.06 — negative prices are not valid for a genuine product sale and likely represent bad-debt adjustments bundled into the transaction log. Quantity contains an extreme value of -80,995 in a single line, well beyond typical return volumes, suggesting either a bulk correction or a data entry error. Both need exclusion or separate handling before any revenue analysis. |
| **Consistency** | 5,268 exact duplicate rows found (same invoice, product, quantity, date, price, and customer). These likely stem from duplicate logging at capture time and should be removed before aggregate analysis to avoid double-counting. |
| **Timeliness** | Not applicable in the traditional sense — this is historical data (2010–2011), not a live feed. Limitation: findings reflect that period's UK retail behavior, not current conditions. |
| **Validity** | A "POST" stock code represents shipping fees, not an actual product, and should be excluded from product-level analysis. Product codes are otherwise consistently formatted. |

## 5. Data Dictionary (Draft)

| Variable | Description | Data Type | Notes |
|---|---|---|---|
| InvoiceNo | Unique transaction ID | object (text) | Prefix "C" = cancellation |
| StockCode | Unique product identifier | object (text) | "POST" = shipping fee, not a product |
| Description | Product name | object (text) | 1,454 missing values |
| Quantity | Units purchased per line item | int64 | Negative values = returns/cancellations; extreme outlier at -80,995 |
| InvoiceDate | Date and time of transaction | datetime64 | Range: Dec 2010 – Dec 2011 |
| UnitPrice | Price per unit (GBP) | float64 | Contains invalid negative values — needs review |
| CustomerID | Unique customer identifier | float64 | 135,080 missing (24.9%) — likely guest checkouts |
| Country | Customer's country | object (text) | No missing values |